# Kepler transit detection: localized instrumental-defect study

This is a source-informed, exploratory ASTR 12600 research experiment. Compare standard Astropy BLS, a near-gap mask, and a persistent negative-step mask on genuine NASA Kepler Q8 PDCSAP flux. Artificial transits are inserted before identical detrending; the detector never receives injection truth. Separate conditions are unmodified observed photometry and synthetic step defects. The step detector is not the published Robnik (2026) or Aigrain (2017) pipeline.

**Interpretation warning:** The inverted/rolled-star proxy null is not a valid estimate of Kepler mission-wide FAP. Do not claim a new planet discovery or numerical superiority to established papers.

## 1. Retrieve current code and record its exact revision

A public internet connection is required for GitHub and NASA MAST downloads. No university credentials are required for these public data.

In [ ]:
import os, subprocess
repo="/content/ASTR_FInal_Project"
if not os.path.isdir(repo+"/.git"):
    subprocess.run(["git","clone","--depth","1",
                    "https://github.com/DrewThomas09/ASTR_FInal_Project.git",repo],check=True)
else:
    subprocess.run(["git","-C",repo,"pull","--ff-only"],check=True)
os.chdir(repo)
print("Commit:",subprocess.check_output(["git","rev-parse","HEAD"],text=True).strip())

## 2. Install astronomy and regression-test packages

In [ ]:
%pip -q install "numpy>=1.26,<3" "pandas>=2,<4" "matplotlib>=3.7,<4" "astropy>=6,<9" "scipy>=1.11,<2" "requests>=2.31,<3" "scikit-learn>=1.3,<2" "pytest>=8,<10"

## 3. Review the unmodified, precommitted study selection

Any change to source stars, artifact parameters, test thresholds or inclusion criteria after observing results is a **development revision**, not untouched external validation.

In [ ]:
import json
from pathlib import Path
cfg=json.loads(Path("LOCALIZED_STEP_STUDY_FROZEN_2026_11_07.json").read_text())
print("Calibration",cfg["calibration_stars"])
print("Heldout",cfg["heldout_stars"])
print("Methods",cfg["methods"])
print("Limits",cfg["preregistration_limitations"])

## 4. Run engineering tests

In [ ]:
subprocess.run(["python","-m","pytest","-q",
                    "tests/test_localized_discontinuity.py",
                    "tests/test_localized_step_ablation.py"],check=True)

## 5. Run the source-grounded observational experiment

The full experiment downloads official FITS files. All missing quarters or failed sources remain logged. Runtime depends on internet and external archive availability.

In [ ]:
subprocess.run(["python","-u","localized_step_ablation.py",
                    "--manifest","LOCALIZED_STEP_STUDY_FROZEN_2026_11_07.json",
                    "--out","results/localized_step_study"],check=True)

## 6. Inspect the results, including failures and masking losses

In [ ]:
from IPython.display import display,Image
import pandas as pd
out=Path("results/localized_step_study")
report=json.loads((out/"summary_and_limits.json").read_text())
display(pd.DataFrame(report["sources"]))
display(pd.DataFrame(report["per_condition_results"]))
display(pd.DataFrame(report["per_condition_proxy_nulls"]))
display(pd.DataFrame(report["paired_results"]))
print("Scientific limitations:")
for x in report["science_limits"]:
    print(" -",x)
for condition in ("no_added_defect","synthetic_spsd_like_step"):
    path=out/f"recovery_{condition}.png"
    if path.exists(): display(Image(filename=str(path)))

## 7. Download the full reproducibility artifact

Includes SHA-256 of each source FITS, frozen thresholds, every injected trial, proxy controls and error records.

In [ ]:
import shutil
from google.colab import files
zip_path=shutil.make_archive("/content/Kepler_Localized_Step_Study",
                              "zip",root_dir="results",
                              base_dir="localized_step_study")
files.download(zip_path)